# Lab de Limpeza - Camada Silver: Produtos

In [1]:
import pandas as pd
import numpy as np
import s3fs

In [2]:
storage_options = {
    "key": "minio",
    "secret": "minio123",
    "client_kwargs": {"endpoint_url": "http://localhost:9000"}
}

In [3]:
fs = s3fs.S3FileSystem(**storage_options)
arquivos = fs.glob("bronze/produtos/**/*.csv")
arquivos

['bronze/produtos/ingestion_date=2026-04-29T00:30:04/produtos.csv']

In [4]:
df = pd.concat(
    [
        pd.read_csv(f"s3://{arquivo}", storage_options=storage_options) 
        for arquivo in arquivos
    ],
    ignore_index=True
)
df.head()

,id_produto,nome_produto,familia,marca,embalagem,volume_ml,preco_tabela,custo_unitario,_ingested_at,_source_file
0,P001,Coca-Cola Original 350ml LT,Refrigerantes,Coca-Cola,Lata,350 ml,4.2,2.10,2026-04-29 00:30:04.531079,produtos.csv
1,P002,Coca-Cola Original 2L PET,Refrigerante,Coca-Cola,PET,2000,9.5,4.50,2026-04-29 00:30:04.531079,produtos.csv
2,P003,Coca-Cola Original 1L PET,Refrigerantes,Coca-Cola,PET,1000,6.8,3.20,2026-04-29 00:30:04.531079,produtos.csv
3,P004,Coca-Cola Zero 350ml LT,Refrigerantes,Coca-Cola,Lata,350 ml,4.2,2.10,2026-04-29 00:30:04.531079,produtos.csv
4,P005,Sprite 350ml LT,Refrigerante,Sprite,Lata,350,3.9,1.95,2026-04-29 00:30:04.531079,produtos.csv


Padronização de família

In [9]:
df["familia"].unique()

<ArrowStringArray>
[  'Refrigerante',           'Suco', 'Bebida Vegetal',      'Isotônico',
     'Energético',           'Água',   'Água de Coco']
Length: 7, dtype: str

In [15]:
df['familia'] = df['familia'].str.encode('latin1').str.decode('utf-8', errors='ignore')
df['nome_produto'] = df['nome_produto'].str.encode('latin1').str.decode('utf-8', errors='ignore')

In [8]:
df['familia'] = df['familia'].str.replace('Refrigerantes', 'Refrigerante').str.strip()

Limpeza do Volume

In [10]:
df['volume_ml'] = df['volume_ml'].astype(str).str.replace(' ml', '', case=False)
df['volume_ml'] = pd.to_numeric(df['volume_ml'])

In [11]:
df

,id_produto,nome_produto,familia,marca,embalagem,volume_ml,preco_tabela,custo_unitario,_ingested_at,_source_file
0,P001,Coca-Cola Original 350ml LT,Refrigerante,Coca-Cola,Lata,350,4.2,2.10,2026-04-29 00:30:04.531079,produtos.csv
1,P002,Coca-Cola Original 2L PET,Refrigerante,Coca-Cola,PET,2000,9.5,4.50,2026-04-29 00:30:04.531079,produtos.csv
2,P003,Coca-Cola Original 1L PET,Refrigerante,Coca-Cola,PET,1000,6.8,3.20,2026-04-29 00:30:04.531079,produtos.csv
3,P004,Coca-Cola Zero 350ml LT,Refrigerante,Coca-Cola,Lata,350,4.2,2.10,2026-04-29 00:30:04.531079,produtos.csv
4,P005,Sprite 350ml LT,Refrigerante,Sprite,Lata,350,3.9,1.95,2026-04-29 00:30:04.531079,produtos.csv
5,P006,Sprite 2L PET,Refrigerante,Sprite,PET,2000,8.9,4.20,2026-04-29 00:30:04.531079,produtos.csv
6,P007,Fanta Laranja 350ml LT,Refrigerante,Fanta,Lata,350,3.9,1.95,2026-04-29 00:30:04.531079,produtos.csv
7,P008,Fanta Uva 350ml LT,Refrigerante,Fanta,Lata,350,3.9,1.95,2026-04-29 00:30:04.531079,produtos.csv
8,P009,Schweppes TÃ´nica 350ml LT,Refrigerante,Schweppes,Lata,350,4.5,2.25,2026-04-29 00:30:04.531079,produtos.csv
9,P010,Del Valle Uva 1L TP,Suco,Del Valle,Tetra,1000,7.2,3.60,2026-04-29 00:30:04.531079,produtos.csv


Tipos numéricos

In [12]:
df['preco_tabela'] = df['preco_tabela'].astype(float)
df['custo_unitario'] = df['custo_unitario'].astype(float)

Feature Engineering

In [13]:
df['margem_unitaria_estimada'] = df['preco_tabela'] - df['custo_unitario']

In [17]:
df_silver = df[[
    'id_produto', 'nome_produto', 'familia', 'marca', 
    'embalagem', 'volume_ml', 'preco_tabela', 'custo_unitario', 
    'margem_unitaria_estimada', '_ingested_at',"_source_file"
]]

In [20]:
try:
    df_silver.to_parquet(
        path="s3://silver/produtos/produtos_clean.parquet",
        storage_options=storage_options,
        index=False,
        compression="snappy"
    )
    print("✓ Tabela de Produtos (sem partições) salva com sucesso!")
except Exception as e:
    print(f"Erro ao salvar na camada Silver: {e}")

✓ Tabela de Produtos (sem partições) salva com sucesso!
